# 11.1 模型登记与交付清单

把 9.1 最终评估 MAE 最低的候选（红酒、白酒各一个随机森林）登记为平台里的模型版本（状态：候选，验收与交付由你决定），
用打分入口 `src/predict.py` 跑一份示例证明它能用。模型不重训；最终评估集不再评估。


In [1]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
STEP = ROOT / "steps/11_部署与交付/11.1_模型登记与交付清单"
OUT = STEP / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
S31 = ROOT / "steps/03_数据划分/3.1_预测任务定义_固定划分与验证基线/outputs"
S71 = ROOT / "steps/07_模型选择与训练/7.1_候选模型比较/outputs"
S91 = ROOT / "steps/09_评估与诊断/9.1_最终评估/outputs"
FEATURES = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
            "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"]
WINES = {"red": "红酒", "white": "白酒"}
final = json.loads((S91 / "final_evaluation.json").read_text(encoding="utf-8"))
chosen = {w: min(final["files"][w]["candidates"], key=lambda c: final["files"][w]["candidates"][c]["mae"]) for w in WINES}
assert all(c == "随机森林" for c in chosen.values()), chosen

run = dsflow.start_run("11.1", project=ROOT, hypothesis="9.1 最终评估 MAE 最低的候选可以原样登记为候选模型，并通过打分入口给新记录估计 quality")
for w, name in WINES.items():
    run.log_input(ROOT / f"data/winequality-{w}.csv", name=name)
run.log_input(S31 / "split_assignments.csv", name="划分表")
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
run.log_params({"登记的候选": {WINES[w]: c for w, c in chosen.items()},
                "模型文件哈希": {WINES[w]: sha(S71 / "models" / f"{w}_{c}.joblib")[:12] for w, c in chosen.items()},
                "最终评估 MAE": {WINES[w]: final["files"][w]["candidates"][c]["mae"] for w, c in chosen.items()},
                "打分入口": "steps/11_部署与交付/11.1_模型登记与交付清单/src/predict.py <red|white> <输入 CSV> <输出 CSV>"})
# 把 9.1 的最终评估指标带进这次运行，交付清单的指标栏才有得填（数值原样取自 9.1 的产物，不重算）
run.log_metrics({f"{WINES[w]}_MAE_最终评估": final["files"][w]["candidates"][c]["mae"] for w, c in chosen.items()}
                | {f"{WINES[w]}_基线_MAE_最终评估": final["files"][w]["baseline"]["mae"] for w in WINES})
print("每类酒登记的候选：", {WINES[w]: c for w, c in chosen.items()})


每类酒登记的候选： {'红酒': '随机森林', '白酒': '随机森林'}


In [2]:
# 打分入口真的跑一遍：取每类酒最终评估集的前 10 行（只留 11 个指标列）当输入，通过子进程调用 predict.py
splits = pd.read_csv(S31 / "split_assignments.csv")
pieces = []
for w, name in WINES.items():
    raw = pd.read_csv(ROOT / f"data/winequality-{w}.csv", sep=";").assign(source_row=lambda d: np.arange(1, len(d) + 1))
    rows = splits[(splits["wine"] == w) & (splits["split"] == "final_evaluation")]["source_row"].head(10)
    sample = raw[raw["source_row"].isin(rows)]
    src = OUT / f"scoring_input_{w}.csv"
    dst = OUT / f"scoring_output_{w}.csv"
    sample[FEATURES].to_csv(src, index=False)
    result = subprocess.run([sys.executable, str(STEP / "src/predict.py"), w, str(src), str(dst)], capture_output=True, text=True, encoding="utf-8")
    print(result.stdout.strip() or result.stderr.strip())
    assert result.returncode == 0, result.stderr
    scored = pd.read_csv(dst)
    assert len(scored) == len(sample) and scored["quality_估计"].notna().all()
    pieces.append(scored.assign(wine=w, source_row=sample["source_row"].to_numpy(), quality=sample["quality"].to_numpy()))
example = pd.concat(pieces, ignore_index=True)[["wine", "source_row", *FEATURES, "quality_估计", "quality"]]
example.to_csv(OUT / "scoring_example.csv", index=False)
run.log_artifact(OUT / "scoring_example.csv", purpose="打分入口对 20 条示例记录（每类酒最终评估集前 10 行）的输出，附真值 quality 仅供对照，不是新的评估", kind="table")
run.log_metrics({"示例打分行数": len(example)})
print(example[["wine", "source_row", "alcohol", "quality_估计", "quality"]].head(6).round(3).to_string(index=False))


red：10 行已打分 → steps\11_部署与交付\11.1_模型登记与交付清单\outputs\scoring_output_red.csv
white：10 行已打分 → steps\11_部署与交付\11.1_模型登记与交付清单\outputs\scoring_output_white.csv
wine  source_row  alcohol  quality_估计  quality
 red           3      9.8       5.146        5
 red           6      9.4       5.271        5
 red           8     10.0       4.980        7
 red          11      9.2       5.266        5
 red          15      9.2       4.938        5
 red          21      9.4       5.185        6


In [3]:
refs = {}
for w, cand in chosen.items():
    path = S71 / "models" / f"{w}_{cand}.joblib"
    refs[w] = run.log_model(path, f"{WINES[w]}_quality_{cand}",
                            description=f"{WINES[w]}的 quality 估计模型：11 个理化指标 → 连续 quality；{cand}，只用训练行拟合；最终评估 MAE {final['files'][w]['candidates'][cand]['mae']:.4f}（基线 {final['files'][w]['baseline']['mae']:.4f}）")
    print(f"登记 {WINES[w]}_quality_{cand} 版本 {refs[w]['version']}（候选）：{path.relative_to(ROOT).as_posix()}")
run.log_artifact(STEP / "src/predict.py", purpose="打分入口：给一份 11 个理化指标的 CSV 输出 quality_估计", kind="other")
conclusion = "；".join(f"{WINES[w]}_quality_{c} {refs[w]['version']} 登记为候选（最终评估 MAE {final['files'][w]['candidates'][c]['mae']:.4f}）" for w, c in chosen.items())
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


登记 红酒_quality_随机森林 版本 8314f626bc19（候选）：steps/07_模型选择与训练/7.1_候选模型比较/outputs/models/red_随机森林.joblib
登记 白酒_quality_随机森林 版本 32bfb8ca1cde（候选）：steps/07_模型选择与训练/7.1_候选模型比较/outputs/models/white_随机森林.joblib
红酒_quality_随机森林 8314f626bc19 登记为候选（最终评估 MAE 0.4963）；白酒_quality_随机森林 32bfb8ca1cde 登记为候选（最终评估 MAE 0.5752）
